In [ ]:
# Cập nhật danh sách gói và cài đặt Python 3.10
!apt-get update -y
!apt-get install python3.10 python3.10-distutils -y

# Tải và cài đặt pip riêng cho Python 3.10
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py

In [ ]:
!git clone https://github.com/deety03503/QG_RSIAT.git /kaggle/working/QG_RSIAT
%cd /kaggle/working/QG_RSIAT

In [ ]:
!python3.10 -m pip install uv
!python3.10 -m uv pip install -q -r kaggle/requirements-kaggle.txt

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
import shutil

dest = Path("/kaggle/working/hf-datasets")
dest.mkdir(parents=True, exist_ok=True)

repos = {
    #"cub": "bentrevett/caltech-ucsd-birds-200-2011",
    "cifar100": "uoft-cs/cifar100",
    #"imagenet_a": "barkermrl/imagenet-a",
    #"imagenet_r": "axiong/imagenet-r",
}

free_gb = shutil.disk_usage("/kaggle/working").free / (1024**3)
print(f"Dung lượng trống: {free_gb:.1f} GB")
if free_gb < 5:
    raise RuntimeError("Dung lượng trống dưới 5 GB; hãy giải phóng thêm dung lượng trước khi tải.")

for name, repo_id in repos.items():
    print(f"\nĐang tải {name}: {repo_id}")
    path = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=str(dest / name),
    )
    print(f"Đã lưu tại: {path}")

print("\nHoàn tất. Thư mục dữ liệu:", dest)

In [ ]:

!python3.10 scripts/launch.py --mode single --config exps/adapter_cifar224.json \
  --data_root /kaggle/working/hf-datasets \
  --out ./out/cifar-qhybrid\
  --aligner qhybrid --lambda_qrel 1.0 --kernel quantum \
  --orth qweighted --rs_kernel quantum \
  --quantum_layers 1 --n_qubits 8